## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 28 samples and 13949 columns.
First few rows of the data:


,overall survival (days),vital status,sample_title,patient id,anti-pd-1 response,study site,gender,age (yrs),disease status,previous mapki,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,607.0,1,Pt1,Pt1,Progressive Disease,UCLA,F,66,M1b,N,...,6.945069,6.289998,7.130597,7.392799,5.839262,5.989320,7.615436,7.070199,5.974466,7.075712
1,927.0,0,Pt2,Pt2,Partial Response,UCLA,M,55,M1c,N,...,8.772441,7.548880,8.507697,9.111232,8.325919,7.400845,9.174003,8.206970,7.369614,8.626720
2,948.0,0,Pt4,Pt4,Partial Response,UCLA,M,62,M1c,Y,...,9.668285,9.305375,9.893391,10.050158,8.971447,8.294603,9.938084,9.148508,8.111535,9.433522
3,439.0,0,Pt5,Pt5,Partial Response,UCLA,M,61,M1c,N,...,4.576620,3.882534,4.736389,5.066071,3.423486,3.613893,5.227428,4.560005,3.456208,4.758636
4,882.0,0,Pt6,Pt6,Partial Response,UCLA,M,51,M1c,Y,...,9.515566,9.599917,10.313973,10.136349,8.642516,8.610092,10.228906,9.957816,8.859417,9.782276


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'sample_title'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
